# Parameter-based Model Pipeline

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import glob
from pprint import pprint

import os
from dataclasses import dataclass
from typing import Dict, List, Tuple
import joblib


In [ ]:
# 配置参数
KEY_NAMES = [
    'TOTUSJH', 'TOTPOT', 'TOTUSJZ', 'ABSNJZH', 'SAVNCPP', 'USFLUX',
    'MEANPOT', 'R_VALUE', 'MEANSHR', 'MEANGAM', 'MEANGBT', 'MEANGBZ',
    'MEANGBH', 'MEANJZD', 'MEANJZH', 'MEANALP'
]

In [ ]:
# 数据清理，删除无效数据以及重复数据
def clean_data(df: pd.DataFrame) -> pd.DataFrame:
    mask = (df[KEY_NAMES].isnull().any(axis=1)) | (df[KEY_NAMES] == 0).all(axis=1)
    df = df[~mask].copy()

    df = df.drop_duplicates(subset=["HARPNUM", "file_name"], keep='first')
    return df

# 数据对齐
def align_df2(df1: pd.DataFrame, df2: pd.DataFrame) -> pd.DataFrame:
    # 对齐数据
    df1 = df1[df1['file_name'].isin(df2['file_name'])]
    df2 = df2[df2['file_name'].isin(df1['file_name'])]

    # 重新索引
    df1 = df1.set_index('file_name').sort_index().reset_index()
    df2 = df2.set_index('file_name').sort_index().reset_index()
    
    return df1, df2

def align_df3(df1: pd.DataFrame, df2: pd.DataFrame, df3: pd.DataFrame) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    # 对齐数据
    df1 = df1[df1['file_name'].isin(df2['file_name']) & df1['file_name'].isin(df3['file_name'])]
    df2 = df2[df2['file_name'].isin(df1['file_name']) & df2['file_name'].isin(df3['file_name'])]
    df3 = df3[df3['file_name'].isin(df1['file_name']) & df3['file_name'].isin(df2['file_name'])]

    # 重新索引
    df1 = df1.set_index('file_name').sort_index().reset_index()
    df2 = df2.set_index('file_name').sort_index().reset_index()
    df3 = df3.set_index('file_name').sort_index().reset_index()
    
    return df1, df2, df3

# 数据预处理
df1 = pd.read_csv('./data/param_data/SHARP_params_new.csv')
df2 = pd.read_csv('./data/param_data/MFR_params.csv')
df3 = pd.read_csv('./data/param_data/PIL_params_new.csv')

df1_clean = clean_data(df1)
df2_clean = clean_data(df2)
df3_clean = clean_data(df3)

# 对齐数据
df1_align2, df2_align2 = align_df2(df1_clean, df2_clean)
df1_align3, df2_align3, df3_align3 = align_df3(df1_clean, df2_clean, df3_clean)

# 输出大小
print(f"df1 shape: {df1_clean.shape}")
print(f"df2 shape: {df2_clean.shape}")
print(f"df3 shape: {df3_clean.shape}")

print(f"df1_align2 shape: {df1_align2.shape}")
print(f"df2_align2 shape: {df2_align2.shape}")
print(f"df1_align3 shape: {df1_align3.shape}")
print(f"df2_align3 shape: {df2_align3.shape}")
print(f"df3_align3 shape: {df3_align3.shape}")

# 保存数据
df1_align2.to_csv('./data/param_data/a2/SHARP_params.csv', index=False)
df2_align2.to_csv('./data/param_data/a2/MFR_params.csv', index=False)
df1_align3.to_csv('./data/param_data/a3/SHARP_params.csv', index=False)
df2_align3.to_csv('./data/param_data/a3/MFR_params.csv', index=False)
df3_align3.to_csv('./data/param_data/a3/PIL_params.csv', index=False)

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import glob
import os
from dataclasses import dataclass
from typing import Dict, List, Tuple
import joblib
from joblib import Parallel, delayed
from datetime import datetime
import warnings

# 配置参数
KEY_NAMES = [
    'TOTUSJH', 'TOTPOT', 'TOTUSJZ', 'ABSNJZH', 'SAVNCPP', 'USFLUX',
    'MEANPOT', 'R_VALUE', 'MEANSHR', 'MEANGAM', 'MEANGBT', 'MEANGBZ',
    'MEANGBH', 'MEANJZD', 'MEANJZH', 'MEANALP'
]

warnings.filterwarnings("ignore", message="CUDA initialization")

@dataclass
class EvaluationResult:
    model: str
    parameters: str
    training_time: float
    accuracy: float
    precision: float
    recall: float
    f1: float
    tss: float
    tn: int
    fp: int
    fn: int
    tp: int

class FCNN(nn.Module):
    def __init__(self, input_size, leaky_alpha, dropout_rate):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_size, 128),
            nn.LeakyReLU(leaky_alpha),
            nn.Dropout(dropout_rate),
            nn.Linear(128, 64),
            nn.LeakyReLU(leaky_alpha),
            nn.Dropout(dropout_rate),
            nn.Linear(64, 1),
            nn.Sigmoid()
        )
    
    def forward(self, x):
        return self.net(x)

def _train_fcnn_wrapper(params):
    try:
        (X_train, y_train, X_val, y_val, X_test, y_test,
         leaky_alpha, dropout_rate, random_state, csv_name, save_dir) = params
        
        start_time = datetime.now()
        device = torch.device('cuda:3' if torch.cuda.is_available() else 'cpu')
        
        # 设置随机种子
        torch.manual_seed(random_state)
        np.random.seed(random_state)
        if device.type == 'cuda':
            torch.cuda.manual_seed_all(random_state)
        
        # 数据转换
        X_train_t = torch.FloatTensor(X_train).to(device)
        y_train_t = torch.FloatTensor(y_train).unsqueeze(1).to(device)
        X_val_t = torch.FloatTensor(X_val).to(device)
        y_val_t = torch.FloatTensor(y_val).unsqueeze(1).to(device)
        X_test_t = torch.FloatTensor(X_test).to(device)
        
        # 模型初始化
        model = FCNN(X_train.shape[1], leaky_alpha, dropout_rate).to(device)
        optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
        criterion = nn.BCELoss()
        
        # 训练过程
        best_loss, no_improve, best_model = float('inf'), 0, None
        train_dataset = TensorDataset(X_train_t, y_train_t)
        train_loader = DataLoader(train_dataset, batch_size=512, shuffle=True)
        
        for epoch in range(100):
            model.train()
            epoch_loss = 0
            for batch_X, batch_y in train_loader:
                optimizer.zero_grad()
                outputs = model(batch_X)
                loss = criterion(outputs, batch_y)
                loss.backward()
                optimizer.step()
                epoch_loss += loss.item()
            
            # 验证
            model.eval()
            with torch.no_grad():
                val_outputs = model(X_val_t)
                val_loss = criterion(val_outputs, y_val_t)
            
            if val_loss < best_loss:
                best_loss, no_improve, best_model = val_loss.item(), 0, model.state_dict()
            else:
                no_improve += 1
                if no_improve >= 10:
                    break
        
        # 评估
        model.load_state_dict(best_model)
        model.eval()
        # 创建模型保存目录
        model_dir = save_dir + "/models"
        os.makedirs(model_dir, exist_ok=True)
        
        # 生成唯一文件名（包含关键参数）
        filename = f"fcnn_{csv_name}_{random_state}_{leaky_alpha:.3f}_{dropout_rate:.1f}.pth"
        model_path = os.path.join(model_dir, filename)
        
        # 保存完整模型信息（包含结构和参数）
        torch.save({
            'model_state_dict': model.state_dict(),
            'input_size': X_train.shape[1],
            'leaky_alpha': leaky_alpha,
            'dropout_rate': dropout_rate,
            'random_state': random_state,
            'val_loss': best_loss
        }, model_path)
        # print(f"模型已保存至：{model_path}")
        with torch.no_grad():
            test_outputs = model(X_test_t)
            y_pred = (test_outputs.cpu().numpy() > 0.5).astype(int).flatten()
        
        tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
        training_time = (datetime.now() - start_time).total_seconds()
        print(f"FCNN训练完成 | random_state={random_state} | leaky alpha={leaky_alpha:.3f} | dropout rate={dropout_rate:.1f} | 耗时: {training_time:.1f}s")
        return EvaluationResult(
            model='FCNN',
            parameters=f"rs={random_state}|alpha={leaky_alpha}|dropout={dropout_rate}",
            training_time=training_time,
            accuracy=accuracy_score(y_test, y_pred),
            precision=precision_score(y_test, y_pred),
            recall=recall_score(y_test, y_pred),
            f1=f1_score(y_test, y_pred),
            tss=(tp/(tp+fn)) - (fp/(fp+tn)),
            tn=tn, fp=fp, fn=fn, tp=tp
        )
    
    except Exception as e:
        print(f"FCNN训练错误: {str(e)}")
        return None

class ModelPipeline:
    def __init__(self, n_jobs: int = -1, save_dir: str = './param/a2', model=None):
        """
        model: "all" 或 None 表示训练所有模型；"rf" / "svm" / "fcnn" 表示只训练对应模型。
        """
        self.scaler = StandardScaler()
        self.dropped_files = []
        self.results: List[EvaluationResult] = []
        self.n_jobs = n_jobs if os.name != 'nt' else 1
        self.save_dir = save_dir
        self.model = (model or "all").lower() if model is not None else "all"
        self._train_all = self.model == "all"

    def get_file_name(self, df):
        df = df.copy()
        df['file_time'] = pd.to_datetime(df['file_time'])
        df['file_name'] = 'hmi.sharp_cea_720s.' + \
                        df['HARPNUM'].astype(str) + '.' + \
                        df['file_time'].dt.strftime('%Y%m%d_%H%M%S') + \
                        '_TAI'
        return df

    def load_data(self, param_file: str) -> Dict[str, pd.DataFrame]:
        # df = self.get_file_name(pd.read_csv(param_file))
        df = pd.read_csv(param_file)
        merged = {}
        for split in ['train', 'valid', 'test']:
            label_df = pd.read_csv(f'./data/split/{split}.csv')
            merged[split] = df.merge(label_df[['file_name', 'label']], on='file_name', how='inner')
            merged[split] = merged[split][['file_name'] + KEY_NAMES + ['label']]
        return merged

    def preprocess(self, data_dict: Dict[str, pd.DataFrame]) -> Dict[str, Dict[str, np.ndarray]]:
        processed = {}
        for split in ['train', 'valid', 'test']:
            df = data_dict[split].copy()
            original_count = len(df)
            mask = (df[KEY_NAMES].isnull().any(axis=1)) | (df[KEY_NAMES] == 0).all(axis=1)
            self.dropped_files.extend(df[mask]['file_name'].tolist())
            df = df[~mask].copy()
            
            if split == 'train':
                df[KEY_NAMES] = self.scaler.fit_transform(df[KEY_NAMES])
            else:
                df[KEY_NAMES] = self.scaler.transform(df[KEY_NAMES])
            
            processed[split] = {
                'X': df[KEY_NAMES].values.astype(np.float32),
                'y': df['label'].values,
                'file_names': df['file_name'].tolist()
            }
            print(f"{split}集: {len(df)}样本 (丢弃{original_count - len(df)})")
        return processed

    def run(self, param_file: str):
        self.results = []
        print(f"\n{'='*40}\n处理 {param_file}\n{'='*40}")
        csv_name = os.path.basename(param_file).replace("_params.csv","")
        
        # 数据加载和预处理
        data = self.load_data(param_file)
        processed = self.preprocess(data)
        
        # 准备数据
        X_train_full = np.concatenate([processed['train']['X'], processed['valid']['X']])
        y_train_full = np.concatenate([processed['train']['y'], processed['valid']['y']])
        X_test = processed['test']['X']
        y_test = processed['test']['y']
        
        # 参数配置
        random_states = [2 ** n for n in range(1, 6)]
        svc_params = {'kernel': ['rbf', 'poly', 'sigmoid'], 'C': [0.1, 1, 100, 1000]}
        rf_params = {'n_estimators': [50, 100, 1000], 'max_depth': [5, 10, 20, None]}
        fcnn_params = {'leaky_alpha': [0, 0.01, 0.1], 'drop_rate': [0.1, 0.3, 0.5, 0.7]}
        
        # 根据 model 参数决定训练哪些模型：None/"all" 训练全部，否则只训练 rf / svm / fcnn 之一
        run_rf = self._train_all or self.model == "rf"
        run_fcnn = self._train_all or self.model == "fcnn"
        run_svm = self._train_all or self.model == "svm"
        
        if run_rf:
            print("\n训练随机森林模型...")
            rf_results = Parallel(n_jobs=self.n_jobs)(
                delayed(self._train_rf)(rs, n_est, depth, X_train_full, y_train_full, X_test, y_test, csv_name)
                for rs in random_states
                for n_est in rf_params['n_estimators']
                for depth in rf_params['max_depth']
            )
            self.results.extend([r for r in rf_results if r])
        
        if run_fcnn:
            print("\n训练FCNN模型...")
            fcnn_tasks = [
                (
                    processed['train']['X'], processed['train']['y'],
                    processed['valid']['X'], processed['valid']['y'],
                    processed['test']['X'], processed['test']['y'],
                    la, dr, rs, csv_name, self.save_dir
                )
                for rs in random_states
                for la in fcnn_params['leaky_alpha']
                for dr in fcnn_params['drop_rate']
            ]
            fcnn_results = Parallel(n_jobs=1 if torch.cuda.is_available() else self.n_jobs)(
                delayed(_train_fcnn_wrapper)(task) for task in fcnn_tasks
            )
            self.results.extend([r for r in fcnn_results if r])

        if run_svm:
            print("\n训练SVM模型...")
            svm_results = Parallel(n_jobs=self.n_jobs)(
                delayed(self._train_svm)(rs, kernel, C, X_train_full, y_train_full, X_test, y_test, csv_name)
                for rs in random_states
                for kernel in svc_params['kernel']
                for C in svc_params['C']
            )
            self.results.extend([r for r in svm_results if r])
        
        # 保存结果和分析
        self._save_results(csv_name)

    def _train_svm(self, rs, kernel, C, X_train, y_train, X_test, y_test, csv_name):
        try:
            start_time = datetime.now()
            model = SVC(kernel=kernel, C=C, probability=True, random_state=rs)
            model.fit(X_train, y_train)
            training_time = (datetime.now() - start_time).total_seconds()
            
            y_pred = model.predict(X_test)
            metrics = self._calc_metrics(y_test, y_pred)
            joblib.dump(model, self.save_dir + f'/models/svm_{csv_name}_{rs}_{kernel}_{C}.pkl')
            
            print(f"SVM训练完成 | random_state={rs} | C={C} | kernel={kernel} | 耗时: {training_time:.1f}s")
            return EvaluationResult(
                model='SVM',
                parameters=f"rs={rs}|kernel={kernel}|C={C}",
                training_time=training_time,
                **metrics
            )
        except Exception as e:
            print(f"SVM训练失败: {str(e)}")
            return None

    def _train_rf(self, rs, n_est, depth, X_train, y_train, X_test, y_test, csv_name):
        try:
            start_time = datetime.now()
            model = RandomForestClassifier(n_estimators=n_est, max_depth=depth, random_state=rs)
            model.fit(X_train, y_train)
            training_time = (datetime.now() - start_time).total_seconds()

            # 构建特征重要性字典
            fi_dict = {
                'model': csv_name,
                'init': f"{rs}_{n_est}_{depth}"
            }
            # 添加每个特征的重要性列
            for feature_name, importance in zip(KEY_NAMES, model.feature_importances_):
                fi_dict[feature_name] = importance

            # 转换为DataFrame并保存
            fi_df = pd.DataFrame([fi_dict])
            fi_df.to_csv(
                self.save_dir + '/rf_if.csv',
                mode='a',          # 追加模式
                header=not os.path.exists(self.save_dir + '/rf_if.csv'),  # 文件不存在时写列名
                index=False
            )

            y_pred = model.predict(X_test)
            metrics = self._calc_metrics(y_test, y_pred)
            joblib.dump(model, self.save_dir + f'/models/rf_{csv_name}_{rs}_{n_est}_{depth}.pkl')

            print(f"RF训练完成 | random_state={rs} | n_est={n_est} | depth={depth} | 耗时: {training_time:.1f}s")
            return EvaluationResult(
                model='RF',
                parameters=f"rs={rs}|n_est={n_est}|depth={depth}",
                training_time=training_time,
                **metrics
            )
        except Exception as e:
            print(f"随机森林训练失败: {str(e)}")
            return None

    def _calc_metrics(self, y_true, y_pred) -> Dict:
        cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
        tn, fp, fn, tp = cm.ravel()
        denom_pos = tp + fn
        denom_neg = fp + tn
        tss = (tp / denom_pos if denom_pos > 0 else 0) - (fp / denom_neg if denom_neg > 0 else 0)
        return {
            'accuracy': accuracy_score(y_true, y_pred),
            'precision': precision_score(y_true, y_pred, zero_division=0),
            'recall': recall_score(y_true, y_pred, zero_division=0),
            'f1': f1_score(y_true, y_pred, zero_division=0),
            'tss': tss,
            'tn': int(tn), 'fp': int(fp), 'fn': int(fn), 'tp': int(tp)
        }

    def _save_results(self, csv_name):

        results_df = pd.DataFrame([r.__dict__ for r in self.results])
        os.makedirs(self.save_dir + '/perf', exist_ok=True)
        # 判断是否需要写入列头
        file_path = self.save_dir + f'/perf/{csv_name}.csv'
        header = not os.path.exists(file_path)
        # 使用追加模式写入
        results_df.to_csv(file_path, mode='w', index=False, header=header)
        print(f"\n结果已保存到 {file_path}")


In [ ]:
# 主程序
params_list_a2 = ["./data/param_data/a2/MFR_params.csv", 
                  "./data/param_data/a2/SHARP_params.csv"]

params_list_a3 = ["./data/param_data/a3/MFR_params.csv",
                  "./data/param_data/a3/SHARP_params.csv",
                  "./data/param_data/a3/PIL_params.csv"]
if __name__ == "__main__":
    pipeline = ModelPipeline(n_jobs=60, save_dir='./param/a2')
    for param_file in params_list_a2:
        pipeline.run(param_file)

    pipeline = ModelPipeline(n_jobs=60, save_dir='./param/a3')
    for param_file in params_list_a3:
        pipeline.run(param_file)


# PII Caculating Pipeline

In [ ]:
import os
import numpy as np
import pandas as pd
import csv
import sys

from astropy.io import fits
from skimage.measure import label, regionprops
from skimage import filters

# -------------------读取数据--------------------------

# 读取 fits 文件
def read_fits(file_name):
    data = fits.open(file_name)[1].data
    return data

# 获取 b_los 和 attr_data 
def get_data(sample):
    data_path = "/data_work/data_sharp_cea/"
    attr_path = "/data_work/mask_data/mfr/"
    harpnum = sample['HARPNUM']
    file_name = sample['file_name']
    file_path = data_path + f'sharp_{harpnum:05d}/{file_name}'

    bl_data = read_fits(file_path + '.magnetogram.fits')
    attr_data = np.load(attr_path + file_name + '.attr.npy')
    return bl_data, attr_data

# 通过 OTSU 对 attr_data 进行阈值分割得到 IoU
def get_iou(heatmap):
    # 计算阈值
    threshold = filters.threshold_otsu(np.abs(heatmap))
    
    # 创建掩膜
    mask = np.zeros_like(heatmap, dtype=int)
    mask[heatmap > threshold] = 1
    mask[heatmap < -threshold] = -1
    
    return mask

# 连通域标记，按面积由大到小排序，并根据激活值的正负给予标记
def get_combined_label(attr_mask):
    combined_label = label(abs(attr_mask), connectivity=2)

    # 将连通域面积按照从大到小排序
    regions = regionprops(combined_label)
    sorted_regions = sorted(regions, key=lambda x: x.area, reverse=True)

    # 赋予编号正负
    re_clabel = np.zeros_like(combined_label)
    for i, region in enumerate(sorted_regions):
        if attr_mask[combined_label == region.label].sum() > 0:
            re_clabel[combined_label == region.label] = i + 1
        else:
            re_clabel[combined_label == region.label] = -1 * (i + 1)

    return re_clabel

def get_combined_list(attr_mask):
    
    combined_masks = get_combined_label(attr_mask)
    
    combined_mask_list = []
    for i in range(combined_masks.min(), combined_masks.max()+1):
        if i != 0 and (combined_masks==i).sum() != 0:
            combined_mask_list.append([i, (combined_masks==i)])

    return combined_mask_list

# 计算 Polarity Imbalance Index
def compute_pii(data, mask, threshold):
    p_data = data > threshold
    n_data = data < -threshold
    # 计算 pii
    flux_num_p = np.sum(p_data & (mask==1))
    flux_num_n = np.sum(n_data & (mask==1))

    with np.errstate(divide='ignore', invalid='ignore'):
        pii = np.abs(flux_num_p - flux_num_n) / (flux_num_p + flux_num_n)

    return pii, flux_num_p, flux_num_n 

# 计算每一连通域的 PII 及相关信息    
def get_conbined_info(blos, combined_mask_list, threshold = 150):

    info_list = []

    for i, region in combined_mask_list:
        # 计算连通域磁通数
        pii, flux_num_p, flux_num_n = compute_pii(blos, region, threshold)

        info_list.append([i, pii, flux_num_p, flux_num_n, (region==1).sum()]) # 保存结果：label，PII，flux_num_p， flux_num_n，size

    return info_list

# 计算总的加权平均 PIIs
def get_weighted_pii(info_list):
    # 转为 numpy array 便于操作
    info_data = np.array(info_list)

    # 分别筛选 i > 0 和 i < 0
    pos_mask = info_data[:,0] > 0
    neg_mask = info_data[:,0] < 0

    # 计算加权平均 PII，权重是面积
    pos_weighted_pii = np.average(info_data[pos_mask, 1], weights=info_data[pos_mask, 4])
    neg_weighted_pii = np.average(info_data[neg_mask, 1], weights=info_data[neg_mask, 4])
    
    return pos_weighted_pii, neg_weighted_pii

# 计算整区域的 PII
def get_entire_pii(blos, mask, threshold = 150):

    pos_entire_pii, pos_flux_num_p, pos_flux_num_n = compute_pii(blos, mask==1, threshold)
    neg_entire_pii, neg_flux_num_p, neg_flux_num_n = compute_pii(blos, mask==-1, threshold)

    return pos_entire_pii, neg_entire_pii, [pos_flux_num_p, pos_flux_num_n, neg_flux_num_p, neg_flux_num_n]

def process_sample(sample, csv_file="./data/PII_records.csv"):
    try:
        # 获取数据
        sample_name = sample['file_name']
        sample_label = sample['label']
        
        blos_data, attr_data = get_data(sample)
        sample_iou = get_iou(attr_data)

        # 连通域标记
        combined_label = get_combined_label(sample_iou)

        # 计算每一连通域的 flux number imbalance ratio
        combined_label_list = get_combined_list(combined_label)
        combined_info_list = get_conbined_info(blos_data, combined_label_list, threshold=150)
        
        # 计算加权平均 PIIs
        pos_weighted_pii, neg_weighted_pii = get_weighted_pii(combined_info_list)
        
        # 计算整区域的 PII
        pos_entire_pii, neg_entire_pii, entire_iou_info = get_entire_pii(blos_data, sample_iou, threshold = 150)
        # # 计算不作 150G 限制的结果

        # 提取关键数据（假设 total_info = [pii_p, pii_n, fluxnum_pp, fluxnum_pn, fluxnum_np, fluxnum_nn]）
        row_data = [
            sample_name,
            sample_label,
            pos_weighted_pii, neg_weighted_pii, 
            pos_entire_pii, neg_entire_pii,
            entire_iou_info, combined_info_list
        ]

        # 追加到 CSV 文件
        with open(csv_file, mode='a', newline='') as f:
            writer = csv.writer(f)
            writer.writerow(row_data)
        
        return row_data
    except Exception as e:
        tqdm.write(f"Error processing {sample['file_name']}: {str(e)}")
        return None

from tqdm.notebook import tqdm

def process_samples_sequential(csv_df, start_idx=0, csv_file="./data/PII_records.csv"):
    """
    非并行处理样本，从指定行开始
    """
    from tqdm.notebook import tqdm  # 使用 Notebook 优化版本
    
    # 初始化 CSV 文件头（如果不存在）
    header = [
        "file_name", "label",
        "weighted pii_pos", "weighted pii_neg",
        "entire pii_pos", "entire pii_neg",
        "entire_iou_info", "combined_info"
    ]
    
    if not os.path.exists(csv_file):
        with open(csv_file, mode='w', newline='') as f:
            writer = csv.writer(f)
            writer.writerow(header)
            f.flush()

    total_records = []
    
    # 创建进度条实例
    pbar = tqdm(total=len(csv_df), initial=start_idx, desc="Processing Samples")
    
    for idx in range(start_idx, len(csv_df)):
        sample = csv_df.iloc[idx]
        sample_name = sample['file_name']
        try:
            # 处理单个样本
            row_data = process_sample(sample, csv_file)
            
            # 更新进度条描述
            pbar.set_description(f"Completed: {sample_name}")
            pbar.update(1)  # 手动更新进度
            
            # 立即刷新输出
            sys.stdout.flush()
            
            total_records.append(row_data)
        except Exception as e:
            tqdm.write(f"Error processing {sample_name} (index: {idx}): {str(e)}")
            continue
            
    pbar.close()  # 处理完成后关闭进度条
    return total_records

import warnings

# Suppress all warnings
warnings.filterwarnings("ignore")

# 使用示例
local_path = "/home/aiia/newDisk/zz_solar/work-parameters/codes/"
input_csv = local_path + '/data/label.csv'
output_csv = local_path + '/data/PII_records.csv'

csv_df = pd.read_csv(input_csv)
results = process_samples_sequential(csv_df, start_idx=0, csv_file=output_csv)
